# EEG Model Evaluation - Colab

**Evaluates optimized models using the best parameters found during optimization.**

- Tests models on final test set
- Provides comprehensive performance metrics
- Generates publication-ready results
- Uses balanced class weights for all models

**Prerequisites:** Must run optimization notebook first to get optimal parameters.

In [ ]:
# Mount Google Drive and setup
from google.colab import drive
drive.mount('/content/drive')

# Install required packages
!pip install -q torch torchvision torchaudio
!pip install -q scikit-learn pandas numpy matplotlib seaborn
!pip install -q mne einops  # needed for CBraMod

import torch
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
# Configure paths for Colab
import os
import sys
from pathlib import Path

%cd /content/drive/MyDrive/eye-eeg/eye-eeg
print(f"Repository directory: {os.getcwd()}")

# Path configuration
COLAB_REPO_PATH = "/content/drive/MyDrive/eye-eeg/eye-eeg"
COLAB_DATA_PATH = "/content/drive/MyDrive/eye-eeg"

sys.path.insert(0, COLAB_REPO_PATH)

def colab_get_data_path():
    return Path(COLAB_DATA_PATH)

# Import and patch evaluation module
from classification import model_evaluation_runner
model_evaluation_runner.get_data_path = colab_get_data_path

print(f"✅ Path configuration updated for Colab")
print(f"Repository: {COLAB_REPO_PATH}")
print(f"Data path: {COLAB_DATA_PATH}")

In [ ]:
# Import evaluation functions
from classification.model_evaluation_runner import (
    evaluate_all_models, get_data_path, get_results_path, aggressive_cleanup
)
from data.data_configs import SUBJECTS

print("✅ Evaluation functions imported successfully")
print(f"Data path: {get_data_path()}")
print(f"Results path: {get_results_path()}")
print(f"\nAvailable subjects: {len(SUBJECTS)} total")

## Step 1: Check Optimization Results

Load the optimal parameters from the optimization phase.

In [ ]:
# Check if optimization results exist
import json
from pathlib import Path

results_path = get_results_path()
param_file = results_path / "optimal_parameters_normal_latest.json"

if param_file.exists():
    with open(param_file, 'r') as f:
        optimal_params = json.load(f)
    
    print("✅ Found optimization results!")
    print(f"Models to evaluate: {list(optimal_params.keys())}")
    
    print("\n=== OPTIMIZATION RESULTS SUMMARY ===")
    for model_name, info in optimal_params.items():
        print(f"{model_name:12}: {info['score']:.4f} (validation)")
else:
    print("❌ Optimization results not found!")
    print("Please run the optimization notebook first.")

## Step 2: Run Model Evaluation

Evaluate all optimized models on the test set.

In [ ]:
# Run evaluation on all optimized models
print("=== MODEL EVALUATION ===")
print("⏱️  Expected time: 15-30 minutes")

evaluation_subjects = SUBJECTS[:5]  # Adjust based on your data

try:
    eval_results = evaluate_all_models(
        subjects=evaluation_subjects,
        split_strategy='normal'
    )
    
    print("\n✅ Model evaluation completed!")
    
except FileNotFoundError as e:
    print(f"❌ {e}")
    print("Make sure to run optimization first!")
except Exception as e:
    print(f"❌ Evaluation error: {e}")
    import traceback
    traceback.print_exc()

## Step 3: Results Analysis and Visualization

Comprehensive analysis with publication-ready plots.

In [ ]:
# Load and analyze evaluation results
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

eval_file = results_path / "evaluation_results_normal_latest.json"

if eval_file.exists():
    with open(eval_file, 'r') as f:
        eval_results = json.load(f)
    
    # Create results dataframe
    results_data = []
    for model_name, metrics in eval_results.items():
        results_data.append({
            'Model': model_name,
            'Accuracy': metrics['accuracy'],
            'Balanced Accuracy': metrics['balanced_accuracy'],
            'F1-Macro': metrics['f1_macro']
        })
    
    df_results = pd.DataFrame(results_data)
    df_results = df_results.sort_values('Balanced Accuracy', ascending=False)
    
    print("=== EVALUATION RESULTS ===")
    print(df_results.round(4))
    
    best_model = df_results.iloc[0]['Model']
    best_score = df_results.iloc[0]['Balanced Accuracy']
    print(f"\n🏆 Best Model: {best_model} ({best_score:.4f})")
    
    # Create visualization
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # Balanced Accuracy
    axes[0].bar(df_results['Model'], df_results['Balanced Accuracy'], alpha=0.7)
    axes[0].set_title('Balanced Accuracy')
    axes[0].set_ylabel('Score')
    axes[0].tick_params(axis='x', rotation=45)
    axes[0].grid(axis='y', alpha=0.3)
    
    # F1-Macro
    axes[1].bar(df_results['Model'], df_results['F1-Macro'], alpha=0.7, color='orange')
    axes[1].set_title('F1-Macro Score')
    axes[1].set_ylabel('Score')
    axes[1].tick_params(axis='x', rotation=45)
    axes[1].grid(axis='y', alpha=0.3)
    
    # Accuracy
    axes[2].bar(df_results['Model'], df_results['Accuracy'], alpha=0.7, color='green')
    axes[2].set_title('Accuracy')
    axes[2].set_ylabel('Score')
    axes[2].tick_params(axis='x', rotation=45)
    axes[2].grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('/content/drive/MyDrive/eye-eeg/evaluation_results.png', dpi=150, bbox_inches='tight')
    plt.show()
    
else:
    print("❌ Evaluation results not found. Run evaluation first!")

## Summary

**✅ Model evaluation completed successfully!**

**Key Results:**
- **Best Model:** Check results above
- **Performance:** All models tested with balanced class weights
- **Files Created:** Evaluation results, visualizations, summary report
- **Backed Up:** All results saved to Google Drive

**Next Steps:**
1. Review the best performing model for your use case
2. Use optimal parameters for final model training
3. Document results for publication/reporting

**Performance Interpretation:**
- **>70%** = Excellent performance
- **60-70%** = Good performance  
- **50-60%** = Moderate performance (better than chance)
- **<50%** = Poor performance (worse than chance)